# Trying to figure out meshing

This is the file used to test iterations of meshing techniques. Code was created with the help of GPT 5.2

In [1]:
import numpy as np
import plotly.graph_objects as go

# Load points
points = np.load("/work/csse463/202620/04/data/ShapeNet/npy/02691156/8bb827904cd9acd36c1cd53dbc9f7b8e.npy")

# Create scatter3d plot
fig = go.Figure(data=[go.Scatter3d(
    x=points[:,0],
    y=points[:,1],
    z=points[:,2],
    mode='markers',
    marker=dict(
        size=2,
        color=points[:,2],  # color by z-value
        colorscale='Viridis',
    )
)])

fig.update_layout(scene=dict(
    xaxis_title='X',
    yaxis_title='Y',
    zaxis_title='Z'
))

fig.show()

# Architecture

In [2]:
print(points.shape)
print(points[:5])

(2048, 3)
[[ 0.15324336  0.0625264  -0.41593865]
 [-0.09700121 -0.01123643 -0.10718381]
 [-0.14571673  0.0498688  -0.12992139]
 [ 0.6149044   0.00507934  0.17888603]
 [-0.14110634 -0.06256618 -0.17233652]]


# Meshing

In [3]:
def local_surface_upsample(pcd, k=6):
    points = np.asarray(pcd.points)
    kdtree = o3d.geometry.KDTreeFlann(pcd)

    new_points = []

    for i, p in enumerate(points):
        _, idx, _ = kdtree.search_knn_vector_3d(p, k)

        for j in idx[1:]:
            midpoint = (p + points[j]) / 2.0
            new_points.append(midpoint)

    all_points = np.vstack([points, np.array(new_points)])

    new_pcd = o3d.geometry.PointCloud()
    new_pcd.points = o3d.utility.Vector3dVector(all_points)

    return new_pcd

In [4]:
def remove_sparse_edges(pcd, nb_neighbors=10, std_ratio=2.0):
    """
    Remove sparse or isolated points (typically at edges or outliers).
    
    Args:
        pcd: Open3D point cloud
        nb_neighbors: number of neighbors to consider
        std_ratio: threshold in standard deviations above mean distance
    
    Returns:
        filtered_pcd: cleaned Open3D point cloud
    """
    cl, ind = pcd.remove_statistical_outlier(nb_neighbors=nb_neighbors,
                                             std_ratio=std_ratio)
    filtered_pcd = pcd.select_by_index(ind)
    return filtered_pcd

In [ ]:
import open3d as o3d

# Convert to Open3D point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

# Fine tuning with relative normal distance based on nearest neigbhor proximity
distances = pcd.compute_nearest_neighbor_distance()
avg_dist = np.mean(distances)

# Adaptive normal estimation
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=2 * avg_dist,
        max_nn=60
    )
)
pcd.orient_normals_consistent_tangent_plane(50)

dense_pcd = local_surface_upsample(pcd, k=6)

print("Upsampled size:", np.asarray(dense_pcd.points).shape)

dense_pcd = dense_pcd.voxel_down_sample(
    voxel_size=avg_dist * 0.5
)

dense_pcd = remove_sparse_edges(dense_pcd, nb_neighbors=10, std_ratio=2.0)

print("Cleaned dense point cloud size:", np.asarray(dense_pcd.points).shape)

pcd = pcd + dense_pcd

# Recompute distances and normals for dense cloud
distances = pcd.compute_nearest_neighbor_distance()
avg_dist = np.mean(distances)

# Adaptive normal estimation (FINE TUNE THIS)
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=4 * avg_dist,
        max_nn=100
    )
)
pcd.orient_normals_consistent_tangent_plane(90)

import plotly.graph_objects as go
import numpy as np

dense_points = np.asarray(dense_pcd.points)

fig = go.Figure(data=[go.Scatter3d(
    x=dense_points[:,0],
    y=dense_points[:,1],
    z=dense_points[:,2],
    mode='markers',
    marker=dict(
        size=1,
        color=dense_points[:,2],
        colorscale='rainbow',
    )
)])

fig.update_layout(
    title="Dense Point Cloud",
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    )
)

fig.show()


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Upsampled size: (12288, 3)
Cleaned dense point cloud size: (6626, 3)


In [ ]:

# Fine tuning with multiple radii
radii = o3d.utility.DoubleVector([
    avg_dist * 4,
    avg_dist * 4.5,
    avg_dist * 5,
    avg_dist * 6,
    avg_dist * 7
    ])

mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
    pcd, radii
)

# Cleanup
mesh = mesh.remove_degenerate_triangles()
mesh = mesh.remove_duplicated_triangles()
mesh = mesh.remove_duplicated_vertices()
mesh = mesh.remove_non_manifold_edges()
mesh.remove_unreferenced_vertices()
mesh = mesh.filter_smooth_laplacian(20)

# Manual boundary detection
from collections import defaultdict

triangles = np.asarray(mesh.triangles)
edge_count = defaultdict(int)

# Count edges
for tri in triangles:
    edges = [tuple(sorted([tri[0], tri[1]])),
             tuple(sorted([tri[1], tri[2]])),
             tuple(sorted([tri[2], tri[0]]))]
    for e in edges:
        edge_count[e] += 1

# Boundary edges
boundary_edges = [e for e, c in edge_count.items() if c == 1]
print("Boundary edges:", len(boundary_edges))

# Build adjacency
adj = defaultdict(list)
for v1, v2 in boundary_edges:
    adj[v1].append(v2)
    adj[v2].append(v1)

# Group into loops
visited = set()
loops = []

for start in adj:
    if start in visited:
        continue
    loop = []
    current = start
    prev = None
    while True:
        loop.append(current)
        visited.add(current)
        neighbors = adj[current]
        next_vertex = None
        for n in neighbors:
            if n != prev:
                next_vertex = n
                break
        if next_vertex is None or next_vertex in visited:
            break
        prev = current
        current = next_vertex
    if len(loop) > 2:
        loops.append(loop)

print("Number of loops:", len(loops))

mesh.compute_vertex_normals()

#------------- Try using Poisson on resampled BPA mesh-------------#
mesh_bpa = mesh
pcd_from_mesh = mesh_bpa.sample_points_poisson_disk(
    number_of_points=200000
)

pcd_from_mesh.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=avg_dist * 4,
        max_nn=100
    )
)

pcd_from_mesh.orient_normals_consistent_tangent_plane(100)

mesh_watertight, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd_from_mesh,
    depth=9,
    scale=1.05,
    linear_fit=True
)

densities = np.asarray(densities)
density_threshold = np.quantile(densities, 0.02)

vertices_to_remove = densities < density_threshold
mesh_watertight.remove_vertices_by_mask(vertices_to_remove)

mesh_watertight = mesh_watertight.filter_smooth_laplacian(1)

mesh = mesh_watertight

# Save
# o3d.io.write_triangle_mesh("reconstructed_mesh.ply", mesh)


Boundary edges: 3521
Number of loops: 289


In [7]:
print("Vertices:", np.asarray(mesh.vertices).shape)
print("Triangles:", np.asarray(mesh.triangles).shape)

Vertices: (136067, 3)
Triangles: (271707, 3)


In [8]:
print(mesh)

TriangleMesh with 136067 points and 271707 triangles.


In [9]:
# print("Watertight:", mesh.is_watertight())
# print("Edge manifold:", mesh.is_edge_manifold())
# print("Vertex manifold:", mesh.is_vertex_manifold())
# print("Self intersecting:", mesh.is_self_intersecting())
# print("Watertight:", mesh.is_watertight())

In [10]:
vertices = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)

new_triangles = []

for loop in loops:
    v0 = loop[0]
    for i in range(1, len(loop) - 1):
        new_triangles.append([v0, loop[i], loop[i+1]])

if new_triangles:
    triangles = np.vstack([triangles, np.array(new_triangles)])
    mesh.triangles = o3d.utility.Vector3iVector(triangles)

mesh.compute_vertex_normals()

TriangleMesh with 136067 points and 273945 triangles.

In [ ]:
# Visualize
import plotly.graph_objects as go

vertices = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)

fig = go.Figure(data=[
    go.Mesh3d(
        x=vertices[:,0],
        y=vertices[:,1],
        z=vertices[:,2],
        i=triangles[:,0],
        j=triangles[:,1],
        k=triangles[:,2],
        color='orange',
        opacity=1.0
    )
])

fig.update_layout(
    scene=dict(
        aspectmode='data',   # NEED THIS - had an issue with visual warping due to unbalanced axes earlier
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False)
    )
)

fig.show()